# House Prices · XGBoost + Lasso với log diện tích

**Giả thuyết:** Dùng log diện tích trong nhánh Lasso có thể giảm lỗi của ensemble 80/20.

### 1. Đọc cấu hình và dữ liệu

Dùng lại cấu hình mô hình và hai bộ fold của Phúc. Target là log tự nhiên của giá bán.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline

cwd = Path.cwd().resolve()
ROOT = next(path for path in (cwd, *cwd.parents)
            if (path / "final_model/config.json").is_file())
HERE = ROOT / "experiments/2026-11-15_Huy_ensemble_log_area"
sys.path.insert(0, str(ROOT))

from final_model.run_model import feature_sets
from src.model import make_lasso, make_xgboost
from src.prepare_data import DATA_DIR, ID, TARGET, make_preprocessor

CONFIG = json.loads((ROOT / "final_model/config.json").read_text(encoding="utf-8"))
SPLITS = {
    "original": "15-11-2026_ensemble_xgboost_lasso_phuc/outputs/20261115_111145_484418",
    "seed123": "15-11-2026_ensemble_seed123_phuc/outputs/20261115_121000_620119",
}
AREA_COLUMNS = ["GrLivArea", "TotalSF"]



train = pd.read_csv(DATA_DIR / "raw/train.csv")
target = np.log(train[TARGET])
output = HERE / "outputs"
output.mkdir(exist_ok=True)
print(f"Train: {len(train)} dòng")

Train: 1460 dòng


### 2. Tạo đặc trưng và mô hình

Áp dụng `log1p` cho `GrLivArea` và `TotalSF` trong Lasso để giảm ảnh hưởng của diện tích rất lớn. Hai phiên bản dùng chung XGBoost và cùng tỷ lệ 80% XGBoost, 20% Lasso trên log giá.

In [2]:
def make_inputs(frame):
    """Apply a deterministic transform shared by training and inference."""
    features = feature_sets(frame, CONFIG)
    log_area = features["lasso"].copy()
    log_area[AREA_COLUMNS] = np.log1p(log_area[AREA_COLUMNS])
    return {**features, "log_area_lasso": log_area}


def training_mask(frame):
    rule = CONFIG["outlier_rule"]
    return ~(frame["GrLivArea"].gt(rule["GrLivArea_gt"])
             & frame[TARGET].lt(rule["SalePrice_lt"]))


def fit_predict(name, features, target, fit_rows, prediction_features):
    estimator = (
        make_xgboost(**CONFIG["xgboost"]["params"])
        if name == "xgboost" else make_lasso(**CONFIG["lasso"]["params"])
    )
    model = make_pipeline(make_preprocessor(features), estimator)
    model.fit(features.loc[fit_rows], target.loc[fit_rows])
    return model.predict(prediction_features).astype(float)


def blend(xgb, lasso):
    weights = CONFIG["weights"]
    return weights["xgboost"] * xgb + weights["lasso"] * lasso



features = make_inputs(train)
eligible = training_mask(train)

### 3. So sánh trên cùng các fold

Fit tiền xử lý và mô hình trong từng fold. Quy tắc lọc ngoại lệ chỉ áp dụng cho phần huấn luyện. Cả 1.460 căn được tính vào RMSE OOF, dựa trên giá dự đoán khi căn đó nằm ngoài tập huấn luyện.

In [3]:
def evaluate(train, features, target, eligible, output):
    scores = []
    for split, source in SPLITS.items():
        reference = pd.read_csv(HERE.parent / source / "validation_predictions.csv")
        folds = train[ID].map(reference.set_index(ID)["fold"])
        if folds.isna().any() or set(folds) != set(range(1, 6)):
            raise ValueError(f"Invalid fold assignment for {split}")
        predictions = pd.DataFrame({ID: train[ID], "fold": folds, "log_price": target})
        for fold in sorted(folds.unique()):
            validate = folds.eq(fold)
            fit_rows = ~validate & eligible
            for name, inputs in features.items():
                predictions.loc[validate, name] = fit_predict(
                    name, inputs, target, fit_rows, inputs.loc[validate]
                )
        predictions["baseline"] = blend(predictions["xgboost"], predictions["lasso"])
        predictions["log_area"] = blend(predictions["xgboost"], predictions["log_area_lasso"])
        predictions.to_csv(output / f"oof_{split}.csv", index=False)
        for scope, rows in [("all", predictions), *predictions.groupby("fold")]:
            baseline = np.sqrt(np.mean((rows["baseline"] - rows["log_price"]) ** 2))
            candidate = np.sqrt(np.mean((rows["log_area"] - rows["log_price"]) ** 2))
            scores.append({"split": split, "scope": scope, "n": len(rows),
                           "baseline": baseline, "log_area": candidate,
                           "delta": candidate - baseline})
    return pd.DataFrame(scores)



metrics = evaluate(train, features, target, eligible, output)
metrics.to_csv(output / "metrics.csv", index=False)
print(metrics.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

   split scope    n  baseline  log_area     delta
original   all 1460  0.130066  0.128776 -0.001290
original     1  292  0.133112  0.132835 -0.000276
original     2  292  0.107599  0.107404 -0.000195
original     3  292  0.171219  0.166511 -0.004708
original     4  292  0.118537  0.118863  0.000326
original     5  292  0.109191  0.109001 -0.000190
 seed123   all 1460  0.128204  0.126913 -0.001291
 seed123     1  292  0.104179  0.104089 -0.000090
 seed123     2  292  0.143036  0.141175 -0.001861
 seed123     3  292  0.117527  0.117866  0.000338
 seed123     4  292  0.126770  0.125893 -0.000876
 seed123     5  292  0.144863  0.141520 -0.003343


**Nhận xét.** RMSE OOF giảm từ **0,13007 xuống 0,12878** trên bộ fold gốc và từ **0,12820 xuống 0,12691** với seed 123. Cấu hình mới cải thiện 4/5 fold trên mỗi bộ chia. Điểm Kaggle chưa được đo.

### 4. Tạo submission

Huấn luyện lại ensemble mới trên tập train sau lọc. Dùng `exp` để đưa dự đoán về giá gốc, sau đó đối chiếu `Id` với file mẫu.

In [4]:
test = pd.read_csv(DATA_DIR / "raw/test.csv")
test_features = make_inputs(test)
xgb, lasso = (
    fit_predict(name, features[name], target, eligible, test_features[name])
    for name in ("xgboost", "log_area_lasso")
)
prices = np.exp(blend(xgb, lasso))
if not np.isfinite(prices).all() or not (prices > 0).all():
    raise ValueError("Submission prices must be finite and positive.")
submission = pd.DataFrame({ID: test[ID], TARGET: prices})
sample = pd.read_csv(DATA_DIR / "raw/sample_submission.csv")
if not submission[ID].equals(sample[ID]):
    raise ValueError("Submission IDs differ from the sample.")
submission.to_csv(output / "submission.csv", index=False)
print(f"Đã lưu {len(submission)} dự đoán vào outputs/submission.csv.")

Đã lưu 1459 dự đoán vào outputs/submission.csv.
